# Send an Outlook Email with Python

https://medium.com/mlearning-ai/use-python-to-send-outlook-emails-d673ce9e33e4

In [1]:
import pandas as pd
import win32com.client as win32
from datetime import datetime
import os
import re

In [2]:
pth = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\PGF UCITS Share Class Hedges'
fls = [int(re.match('\d{8}', str(k)).group()) for k in os.listdir(pth) if re.match('\d{8}', str(k)) != None]
fln = os.path.join(pth, str(max(fls)) + ' PGF Share Class Hedges.xlsx')
#fln = os.path.join(pth, f'{rptDate.strftime("%Y%m%d")} PGF Share Class Hedges.xlsx')

In [3]:
rptDate = datetime.strptime(str(max(fls)), "%Y%m%d")
rptDate

datetime.datetime(2023, 10, 23, 0, 0)

In [4]:
rptDate.strftime("%A %#d %B %Y")

'Monday 23 October 2023'

In [5]:
#fln = os.path.join(pth, '20230906 PGF Share Class Hedges.xlsx')
fln = os.path.join(pth, '20231005 PGF Share Class Hedges.xlsx')
df = pd.read_excel(fln, sheet_name = 'Summary', header = 0, nrows = 7, usecols = 'A, I')  # read today's summary
msg = df[(df.iloc[:,1] > 1.05) | (df.iloc[:,1] < 0.95)]                                   # filter for excesses
msg['o_u'] = msg.iloc[:,1].apply(lambda x: 'EXCEEDS' if (x > 1.05) else 'FALLS SHORT OF') # new column

In [6]:
if len(msg) > 0:
    message = ''
    for ind in msg.index:
        message = message + ' ' + str(msg['Class'][ind]) + ' ' + str(msg.iloc[:,2][ind])
    message = message + ' THE REGULATORY LIMIT'
else: 
    message = 'ALL HEDGES ARE WITHIN THE REGULATORY LIMITS'

message

'ALL HEDGES ARE WITHIN THE REGULATORY LIMITS'

In [7]:
outlook      = win32.Dispatch('outlook.application') # specify Outlook application
mail         = outlook.CreateItem(0) # create the mail object
#mail.Subject = 'Credit Ratings from PFS for ' + datetime.now().strftime('%#d %b %Y %H:%M')
mail.Subject = f'PGF Share Class Hedging - {rptDate.strftime("%A %#d %B %Y")} - {message}'
mail.To      = 'hilton.netta@prescient.co.za'
#mail.To      = 'multiasset@prescient.co.za; PIMEquity@prescient.co.za; fixedinterest@prescient.co.za'
#mail.CC      = 'nasreen.hisham@prescient.co.za>; ronel.sindo@prescient.co.za; nazley.herandien@prescient.co.za' 
mail.BCC     = ''
#attachment   = mail.Attachments.Add(fl)
#attachment.PropertyAccessor.SetProperty("http://schemas.microsoft.com/mapi/proptag/0x3712001F", "doggy_img")
mail.HTMLBody = rf"""
Please see below PIM's Prescient Global Funds share class hedges for {rptDate.strftime("%A %#d %B %Y")}.
"""
#mail.Attachments.Add(fl) # attaching a file
#cid = content id. By specifying it to be currency_img, we can show the image in the Email body.

In [8]:
mail.Send()